In [1]:
import pandas as pd
df = pd.read_csv("IGCC_Uncertainty_dataset.csv")

In [2]:
df

,Ambient Air Temperature (Dry Bulb Average) (°F),As-Delivered Coal Cost ($/ton),CO2 Removal Efficiency (%),Power Requirement (% MWg),Cooling Water Temperature Drop (°F),Plant Outputs: Captured CO2: Flow Rate (tons/hr),Performance Parameter: Net Electrical Output (MW),Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton)
0,65.849998,58.299999,60.787498,8.295,28.900000,350.925293,566.327698,206.435303
1,80.550003,55.700001,85.112503,9.285,25.500000,463.870056,518.344238,150.798462
2,64.949997,62.900002,62.187500,6.435,47.299999,358.609619,578.199097,203.933243
3,37.950001,48.700001,86.512497,9.015,11.700000,496.849060,547.814941,144.885849
4,69.150002,31.500000,63.937500,6.495,37.500000,365.735321,572.414734,179.622055
...,...,...,...,...,...,...,...,...
195,32.549999,58.700001,69.362503,7.215,38.099998,411.700623,586.113220,179.689514
196,64.050003,51.099998,84.762497,9.555,41.500000,472.210388,528.255981,147.134338
197,43.650002,68.699997,65.512497,8.955,40.900002,385.882538,570.839600,196.145859
198,34.049999,49.900002,62.537498,11.565,17.900000,374.430664,562.361938,194.301773


In [3]:
df.isna().sum()

Ambient Air Temperature (Dry Bulb Average) (°F)                  0
As-Delivered Coal Cost ($/ton)                                   0
CO2 Removal Efficiency (%)                                       0
Power Requirement (% MWg)                                        0
Cooling Water Temperature Drop (°F)                              0
Plant Outputs: Captured CO2: Flow Rate (tons/hr)                 0
Performance Parameter: Net Electrical Output (MW)                0
Costs of CO2 Avoided & Captured: Cost of CO2 Captured ($/ton)    0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
x = df[['Ambient Air Temperature (Dry Bulb Average) (°F)',
    'As-Delivered Coal Cost ($/ton)',
    'CO2 Removal Efficiency (%)',
    'Power Requirement (% MWg)',
    'Cooling Water Temperature Drop (°F)']]

y = df['Performance Parameter: Net Electrical Output (MW)']

In [6]:
x

,Ambient Air Temperature (Dry Bulb Average) (°F),As-Delivered Coal Cost ($/ton),CO2 Removal Efficiency (%),Power Requirement (% MWg),Cooling Water Temperature Drop (°F)
0,65.849998,58.299999,60.787498,8.295,28.900000
1,80.550003,55.700001,85.112503,9.285,25.500000
2,64.949997,62.900002,62.187500,6.435,47.299999
3,37.950001,48.700001,86.512497,9.015,11.700000
4,69.150002,31.500000,63.937500,6.495,37.500000
...,...,...,...,...,...
195,32.549999,58.700001,69.362503,7.215,38.099998
196,64.050003,51.099998,84.762497,9.555,41.500000
197,43.650002,68.699997,65.512497,8.955,40.900002
198,34.049999,49.900002,62.537498,11.565,17.900000


In [7]:
y

0      566.327698
1      518.344238
2      578.199097
3      547.814941
4      572.414734
          ...    
195    586.113220
196    528.255981
197    570.839600
198    562.361938
199    556.459290
Name: Performance Parameter: Net Electrical Output (MW), Length: 200, dtype: float64

In [8]:
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("\n=== REGRESSION MODELS ===\n")


=== REGRESSION MODELS ===



In [9]:
# Linear Regression
reg_pipe = Pipeline([
    ('scaler',StandardScaler()),
    ('reg',LinearRegression())
])
reg_grid = GridSearchCV(reg_pipe, {}, cv=5, scoring='r2', n_jobs=-1)
reg_grid.fit(x_train,y_train)
print("Linear Regression best params:",reg_grid.best_params_)

best_reg = reg_grid.best_estimator_
print('Train R2:',best_reg.score(x_train,y_train))
print('Test R2:',best_reg.score(x_test,y_test))

Linear Regression best params: {}
Train R2: 0.9996642926183138
Test R2: 0.9996839186465081


In [10]:
# SVR
svr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR())
])
svr_grid = GridSearchCV(
    svr_pipe,
    {
        "svr__kernel": ["linear", "rbf"],
        "svr__C": [0.01, 0.1, 1, 10,100],
        "svr__gamma": ["scale"]
    },
    cv=5, scoring="r2", n_jobs=-1
)
svr_grid.fit(x_train, y_train)

best_svr = svr_grid.best_estimator_
print("SVR best params:", svr_grid.best_params_)
print('Train R2:',best_svr.score(x_train,y_train))
print('Test R2:',best_svr.score(x_test,y_test))

SVR best params: {'svr__C': 10, 'svr__gamma': 'scale', 'svr__kernel': 'linear'}
Train R2: 0.9996563778875543
Test R2: 0.9996752467632783


In [11]:
# XGBoost Regressor
xgb_grid = GridSearchCV(
    XGBRegressor(random_state=42),
    {
        "n_estimators": [100, 300],
        "max_depth": [3, 6, 10],
        "learning_rate": [0.01, 0.1, 0.3],
        "subsample": [0.8, 1.0],
        "colsample_bytree": [0.8, 1.0]
    },
    cv=5, scoring="r2", n_jobs=-1
)
xgb_grid.fit(x_train, y_train)

best_xgb = xgb_grid.best_estimator_
print("XGBoost best params:", xgb_grid.best_params_)
print('Train R2:',best_xgb.score(x_train,y_train))
print('Test R2:',best_xgb.score(x_test,y_test))

XGBoost best params: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'subsample': 0.8}
Train R2: 0.9999532503925318
Test R2: 0.9578069323449157


In [12]:
# Gradient Boosting Regressor
gbr_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    {
        "n_estimators": [100, 300],
        "max_depth": [3, 6, 10],
        "learning_rate": [0.01, 0.1, 0.3],
        "subsample": [0.8, 1.0],
        "max_features": [0.8, 1.0]
    },
    cv=5, scoring="r2", n_jobs=-1
)
gbr_grid.fit(x_train, y_train)

best_gbr = gbr_grid.best_estimator_
print("Gradient Boosting best params:", gbr_grid.best_params_)
print('Train R2:', best_gbr.score(x_train, y_train))
print('Test R2:', best_gbr.score(x_test, y_test))

Gradient Boosting best params: {'learning_rate': 0.1, 'max_depth': 3, 'max_features': 0.8, 'n_estimators': 300, 'subsample': 0.8}
Train R2: 0.9999718763513508
Test R2: 0.9283270772807102


In [13]:
# Random Forest Regressor
rfr_grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    {
        "n_estimators": [100,300],
        "max_depth": [None, 5, 10, 20],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ['sqrt',0.5, 0.7]
    },
    cv=5, scoring="r2", n_jobs=-1
)
rfr_grid.fit(x_train, y_train)

best_rfr = rfr_grid.best_estimator_
print("RF best params:", rfr_grid.best_params_)
print('Train R2:',best_rfr.score(x_train,y_train))
print('Test R2:',best_rfr.score(x_test,y_test))

RF best params: {'max_depth': None, 'max_features': 0.7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Train R2: 0.9895487522960531
Test R2: 0.8707550054395895


In [14]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

poly_pipe = Pipeline([
    ('poly',   PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg',    Ridge())
])
poly_grid = GridSearchCV(
    poly_pipe,
    {'poly__degree': [2, 3, 4],
     'reg__alpha':  [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='r2', n_jobs=-1
)
poly_grid.fit(x_train, y_train)

best_poly = poly_grid.best_estimator_
print("Polynomial best params:", poly_grid.best_params_)
print('Train R2:', best_poly.score(x_train, y_train))
print('Test R2:', best_poly.score(x_test, y_test))

Polynomial best params: {'poly__degree': 2, 'reg__alpha': 0.001}
Train R2: 0.9999998800036806
Test R2: 0.9999997554244355


In [15]:
#Checking the RMSE and MAPE scores
from sklearn.metrics import mean_squared_error
import numpy as np

print("\n=== RMSE & MAPE ===\n")

models = {
    'Linear Regression': best_reg,
    'SVR':               best_svr,
    'Random Forest':     best_rfr,
    'XGBoost':           best_xgb,
    'Gradient Boost':    best_gbr,
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mape   = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    print(f"{name} | RMSE: {rmse:.4f} | MAPE: {mape:.4f}%")


=== RMSE & MAPE ===

Linear Regression | RMSE: 0.3285 | MAPE: 0.0484%
SVR | RMSE: 0.3330 | MAPE: 0.0502%
Random Forest | RMSE: 6.6437 | MAPE: 0.9840%
XGBoost | RMSE: 3.7960 | MAPE: 0.5665%
Gradient Boost | RMSE: 4.9474 | MAPE: 0.6754%
Polynomial | RMSE: 0.0091 | MAPE: 0.0014%


In [16]:
#Exporting to excel
import os
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

file       = 'IGCC_Net_Electrical_Output.xlsx'
experiment = 'Target_2_IGCC_Net_Electrical_Output'   # change per experiment

results = []

models = {
    'Linear Regression': best_reg,
    'SVR':               best_svr,
    'Random Forest':     best_rfr,
    'XGBoost':           best_xgb,
    'Gradient Boost':    best_gbr,
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    results.append({
        'Experiment': experiment,
        'Model':      name,
        'Train R2':   round(model.score(x_train, y_train), 4),
        'Test R2':    round(model.score(x_test, y_test), 4),
        'MAE':        round(mean_absolute_error(y_test, y_pred), 4),
        'MSE':        round(mean_squared_error(y_test, y_pred), 4),
        'RMSE':       round(np.sqrt(mean_squared_error(y_test, y_pred)), 4),
        'MAPE (%)':   round(np.mean(np.abs((y_test - y_pred) / y_test)) * 100, 4)
    })

results_df = pd.DataFrame(results)

if os.path.exists(file):
    existing = pd.read_excel(file)
    combined = pd.concat([existing, results_df], ignore_index=True)
    combined.to_excel(file, index=False)
else:
    results_df.to_excel(file, index=False)

print(results_df)

                            Experiment              Model  Train R2  Test R2  \
0  Target_2_IGCC_Net_Electrical_Output  Linear Regression    0.9997   0.9997   
1  Target_2_IGCC_Net_Electrical_Output                SVR    0.9997   0.9997   
2  Target_2_IGCC_Net_Electrical_Output      Random Forest    0.9895   0.8708   
3  Target_2_IGCC_Net_Electrical_Output            XGBoost    1.0000   0.9578   
4  Target_2_IGCC_Net_Electrical_Output     Gradient Boost    1.0000   0.9283   
5  Target_2_IGCC_Net_Electrical_Output         Polynomial    1.0000   1.0000   

      MAE      MSE    RMSE  MAPE (%)  
0  0.2645   0.1079  0.3285    0.0484  
1  0.2736   0.1109  0.3330    0.0502  
2  5.3362  44.1382  6.6437    0.9840  
3  3.0636  14.4093  3.7960    0.5665  
4  3.6547  24.4769  4.9474    0.6754  
5  0.0079   0.0001  0.0091    0.0014  
